In [1]:
import os, copy
import numpy as np
import jax
import jax.numpy as jnp
from jax import vmap
import matplotlib.pyplot as plt
import pandas as pd
import interpax
from scipy.interpolate import interp1d

from jax_cosmo import Cosmology
import jax_cosmo.background as bkgrd
from jax_cosmo.background import radial_comoving_distance

from godmax.get_Cls  import get_Cl
from godmax.get_covs import get_cov
from godmax.base_class import get_vmapped_func_warg
from godmax.base_class import base_class
from godmax.get_radial_profiles import Profiles
from godmax.get_Pkzs import get_Pkz

from tSZxEuclid.godmax.config.loader import merge_params
from tSZxEuclid.godmax.sampling.ell  import setup_ell_arrays

import diag_utils

/home/vtinnane/micromamba/envs/tSZxEuclid-godmax/lib/python3.11/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


In [2]:
from pathlib import Path

# ── Anchor: directory containing this notebook ───────────────────────────────
HERE = Path().resolve()          
DEVELOP = HERE.parents[2]        

# ── Parameter files (diagnostics/params/) ────────────────────────────────────
PARAM_DIR = HERE / 'params'
sim_params_dict, halo_params_dict, analysis_dict, other_params_dict = merge_params(
    str(PARAM_DIR / 'params_default.yaml'),
    str(PARAM_DIR / 'params.yaml'),
)

halo_params_dict['lowpass_Pmm1h_lowk'] = True
halo_params_dict['kthresh_lowpass_Pmm1h_lowk'] = 0.01  # k in h/Mpc below which 1h is suppressed                          
analysis_dict['is_cmb_lensing'] = True          

# ── Cosmology (jax_cosmo) ────────────────────────────────────────────────────
cosmo_params = sim_params_dict['cosmo']
cosmo_jax = Cosmology(
    Omega_c  = cosmo_params['Om0'] - cosmo_params['Ob0'],
    Omega_b  = cosmo_params['Ob0'],
    h        = cosmo_params['H0'] / 100.,
    sigma8   = cosmo_params['sigma8'],
    n_s      = cosmo_params['ns'],
    Omega_k  = 0.,
    w0       = cosmo_params.get('w0', -1.0),
    wa       = 0.,
)

# ── ell array (mutates analysis_dict and halo_params_dict in place) ──────────
setup_ell_arrays(analysis_dict, halo_params_dict, lmin=80.0, lmax=3000.0)
print(f"nell = {len(analysis_dict['l_array_survey'])}, "
      f"ell range [{float(analysis_dict['l_array_survey'][0]):.1f}, "
      f"{float(analysis_dict['l_array_survey'][-1]):.1f}]")

# ── Galaxy n(z) paths ────────────────────────────────────────────────────────
DNDZ_DIR = DEVELOP / 'tSZxEuclid' / 'Data' / 'dndz'

def to_comoving_nz(z_array, nz_array, cosmo):
    """Convert non-normalised n(z) [#gal/z] to comoving n(z) [(Mpc/h)^-3]."""
    a = 1.0 / (1.0 + z_array)
    chi = radial_comoving_distance(cosmo, a)
    dchi_dz = (2.998e5) / bkgrd.H(cosmo, a)   # c [km/s] / H(a) [km/s/(Mpc/h)]
    dV_dzdOmega = chi**2 * dchi_dz
    return nz_array / dV_dzdOmega

dndz_gal = np.loadtxt(DNDZ_DIR / 'nz_euclid_forecast_bin-all.txt')
z_gal  = dndz_gal[:, 0]
nz_gal = dndz_gal[:, 1]
nz_gal_comoving = to_comoving_nz(z_gal, nz_gal, cosmo_jax)
analysis_dict['nbar_gal_comoving_zarray'] = z_gal
analysis_dict['nbar_gal_comoving_val']    = nz_gal_comoving

# ── Lens N(z) bins ───────────────────────────────────────────────────────────
nbins_lens   = 5
nz_lens      = {}
nz_info_dict = {'nbins_lens': nbins_lens}

for nbin in range(nbins_lens):
    d = np.loadtxt(DNDZ_DIR / f'nz_norm_euclid_forecast_bin-{nbin+1}.txt')
    nz_lens[nbin]              = d[:, 1]
    nz_info_dict[f'nz{nbin}'] = nz_lens[nbin]
z_lens = d[:, 0]
nz_info_dict['z_array_lens'] = z_lens
analysis_dict['nz_lens_info_dict'] = nz_info_dict

# ── Source N(z) bins (same as lens for mock) ─────────────────────────────────
nbins_source = 5
nz_source  = {}
nz_info_src = {'nbins': nbins_source}

for nbin in range(nbins_source):
    d = np.loadtxt(DNDZ_DIR / f'nz_norm_euclid_forecast_bin-{nbin+1}.txt')
    nz_source[nbin]             = d[:, 1]
    nz_info_src[f'nz{nbin}']  = nz_source[nbin]
z_source = d[:, 0]
nz_info_src['z_array_source'] = z_source
analysis_dict['nz_source_info_dict'] = nz_info_src

# ── Per-bin nbar [gal/arcmin²] ───────────────────────────────────────────────
deg2_to_arcmin2 = 3600.0
nz_gal_interp   = interp1d(z_gal, nz_gal, bounds_error=False, fill_value=0.0)
nz_gal_on_zlens = nz_gal_interp(z_lens)

pz_bins = np.vstack([nz_lens[jb] for jb in range(nbins_lens)])   # (nbins, Nz)
pz_sum  = np.sum(pz_bins, axis=0)
nbar_lens_bins = np.zeros(nbins_lens)
for jb in range(nbins_lens):
    frac_j = np.zeros_like(z_lens)
    valid  = pz_sum > 0
    frac_j[valid] = pz_bins[jb, valid] / pz_sum[valid]
    nbar_lens_bins[jb] = np.trapezoid(nz_gal_on_zlens * frac_j, z_lens) / deg2_to_arcmin2
analysis_dict['nbar_lens_bins'] = nbar_lens_bins

# ── SO LAT noise ─────────────────────────────────────────────────────────────
SO_NOISE = DEVELOP / 'tSZxEuclid' / 'Data' / 'SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt'
analysis_dict['yy_noise_ell_fname'] = str(SO_NOISE)

# ── Output directory for figures ─────────────────────────────────────────────
FIGDIR = HERE / 'plots'
FIGDIR.mkdir(exist_ok=True)

print('nbar_lens_bins [gal/arcmin²]:', nbar_lens_bins)
print(f"Halo mass range: [{halo_params_dict['lg10_Mmin']}, {halo_params_dict['lg10_Mmax']}], nM={halo_params_dict['nM']}")
print(f"z grid: [{halo_params_dict['zmin']}, {halo_params_dict['zmax']}], nz={halo_params_dict['nz']}")
print(f"PARAM_DIR : {PARAM_DIR}")
print(f"DNDZ_DIR  : {DNDZ_DIR}")
print(f"FIGDIR    : {FIGDIR}")


ERROR:2026-04-13 14:54:19,211:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/vtinnane/micromamba/envs/tSZxEuclid-godmax/lib/python3.11/site-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/vtinnane/micromamba/envs/tSZxEuclid-godmax/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/vtinnane/micromamba/envs/tSZxEuclid-godmax/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to

nell = 15, ell range [90.4, 2269.7]
nbar_lens_bins [gal/arcmin²]: [0.91251697 0.8918078  0.8918939  0.93620033 1.58377251]
Halo mass range: [11.5, 15.5], nM=24
z grid: [0.01, 1.6], nz=22
PARAM_DIR : /home/vtinnane/Documents/Codes/develop/GODMAX/notebooks/diagnostics/params
DNDZ_DIR  : /home/vtinnane/Documents/Codes/develop/tSZxEuclid/Data/dndz
FIGDIR    : /home/vtinnane/Documents/Codes/develop/GODMAX/notebooks/diagnostics/plots


In [3]:
base_test = base_class(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict)
profiles_test = Profiles(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict, base_class_obj=base_test)
Pkz_test = get_Pkz(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict, Profiles_obj=profiles_test)
Cls_test = get_Cl(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict, Pkz_obj=Pkz_test)
cov_test = get_cov(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict, Cls_test)

z_th = np.array(Cls_test.z_array)
nz = len(z_th)
z_sel = np.unique([0, nz//4, nz//2, 3*nz//4, nz-1])
ell_th = np.array(analysis_dict['l_array_survey'])

Loaded yy noise from file; Cl_y_y_tot_mat = theory + noise


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Low-level helpers
# ─────────────────────────────────────────────────────────────────────────────

def _sym(M):
    M = np.asarray(M, dtype=float)
    return 0.5 * (M + M.T)


def _cov_to_corr(C):
    C = _sym(C)
    var = np.diag(C)
    if np.any(var <= 0):
        raise ValueError(f"Non-positive diagonal: min={var.min():.3e}")
    s = np.sqrt(var)
    R = C / np.outer(s, s)
    np.fill_diagonal(R, 1.0)
    return _sym(R)


def _eig_stats(M, tol=0.0):
    """Return (evals, evecs, lambda_min, n_neg) for a symmetric matrix."""
    ev, ec = np.linalg.eigh(_sym(M))
    return ev, ec, float(ev.min()), int((ev < -tol).sum())


def _worst_mode(ev, ec):
    """Describe the most-negative eigenvector."""
    idx = int(np.argmin(ev))
    v = ec[:, idx]
    p = v ** 2
    dom = int(np.argmax(p))
    ipr = float(np.sum(p ** 2))
    return {
        "lam":       float(ev[idx]),
        "dom_idx":   dom,
        "dom_frac":  float(p[dom]),
        "eff_supp":  float(1.0 / ipr) if ipr > 0 else np.inf,
        "vec":       v,
    }


def _build_ng(cov_obj, diag_utils, probes_arr, warn_missing=False):
    """Return (cov_mat, count_per_probe, nell) for the NG component."""
    out = diag_utils.build_cov_matrix(
        cov_obj.covNG_dict, cov_obj.Cl_result_dict,
        probes_arr, warn_missing=warn_missing,
    )
    return np.array(out[0], dtype=float), out[2], out[3]


def _dup_values(arr, atol=0.0, rtol=0.0):
    """Find repeated values in a 1-D array; skip trivially all-same arrays."""
    arr = np.asarray(arr, dtype=float).ravel()
    if np.allclose(arr, arr[0], atol=atol, rtol=rtol):
        return []          # all identical — not interesting
    used = np.zeros(arr.size, dtype=bool)
    groups = []
    for i in range(arr.size):
        if used[i]:
            continue
        grp = [i]
        for j in range(i + 1, arr.size):
            if not used[j] and np.isclose(arr[i], arr[j], atol=atol, rtol=rtol):
                grp.append(j)
        if len(grp) > 1:
            groups.append(grp)
            for k in grp:
                used[k] = True
    return groups


# ─────────────────────────────────────────────────────────────────────────────
# Main diagnostic: one probe set, one component
# ─────────────────────────────────────────────────────────────────────────────

def _diagnose_one(probes_arr, cov_dict, cl_dict, diag_utils,
                  eig_tol=0.0, dup_atol=0.0, dup_rtol=0.0,
                  warn_missing=False):
    """Return a result dict for one (probe_set, component) pair."""
    out      = diag_utils.build_cov_matrix(cov_dict, cl_dict, probes_arr,
                                           warn_missing=warn_missing)
    cov_mat  = _sym(np.array(out[0], dtype=float))
    dv       = out[1]  # entries: list of (probe, bin1, bin2) tuples — not numeric

    ev_c, ec_c, lmin_c, nneg_c = _eig_stats(cov_mat, eig_tol)
    wm_c = _worst_mode(ev_c, ec_c)

    try:
        corr             = _cov_to_corr(cov_mat)
        ev_r, ec_r, lmin_r, nneg_r = _eig_stats(corr, eig_tol)
        wm_r             = _worst_mode(ev_r, ec_r)
        corr_err         = None
    except Exception as e:
        corr = ev_r = ec_r = wm_r = None
        lmin_r = np.nan; nneg_r = 0; corr_err = str(e)

    dup_diag = _dup_values(np.diag(cov_mat), dup_atol, dup_rtol)
    dup_dv   = []  # entries are (probe, bin1, bin2) tuples — not numeric

    return dict(
        shape=cov_mat.shape, n=cov_mat.shape[0],
        min_diag=float(np.min(np.diag(cov_mat))),
        max_asym=float(np.max(np.abs(cov_mat - cov_mat.T))),
        lmin_cov=lmin_c, nneg_cov=nneg_c, wm_cov=wm_c,
        lmin_corr=lmin_r, nneg_corr=nneg_r, wm_corr=wm_r,
        corr_err=corr_err,
        dup_diag=dup_diag, dup_dv=dup_dv,
        cov_mat=cov_mat, corr_mat=corr, dv=dv,
        evals_cov=ev_c, evecs_cov=ec_c,
        evals_corr=ev_r, evecs_corr=ec_r,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Top-level: many probe sets
# ─────────────────────────────────────────────────────────────────────────────

def diagnose_many_probe_sets(
    probe_sets,
    cov_obj,
    diag_utils,
    use_components=("tot", "G", "NG"),
    warn_missing=False,
    eig_tol=0.0,
    dup_atol=0.0,
    dup_rtol=0.0,
    show_matrix=False,
    show_plots=False,
):
    """
    Run eigenvalue diagnostics for every (probe_set, component) pair.

    Prints a compact block per probe set as it runs, then a clean summary
    table at the end.

    Returns nested dict: all_results[label][comp] = result_dict
    """
    comp_dict_attr = {"tot": "covtot_dict", "G": "covG_dict", "NG": "covNG_dict"}
    cl_dict = cov_obj.Cl_result_dict

    all_results = {}
    summary_rows = []      # accumulate for final table

    _NEG = "\033[91m"      # red
    _OK  = "\033[92m"      # green
    _RST = "\033[0m"

    def _flag(lmin_r, nneg_r):
        if np.isnan(lmin_r) or nneg_r > 0 or lmin_r < 0:
            return _NEG + " !" + _RST
        return ""

    for probes_arr in probe_sets:
        label = "+".join(probes_arr)
        res   = {}

        # ── compact per-probe header ──────────────────────────────────────────
        print(f"\n── {label} ", end="")

        for comp in use_components:
            cov_dict = getattr(cov_obj, comp_dict_attr[comp])
            r = _diagnose_one(
                probes_arr, cov_dict, cl_dict, diag_utils,
                eig_tol=eig_tol, dup_atol=dup_atol, dup_rtol=dup_rtol,
                warn_missing=warn_missing,
            )
            res[comp] = r

            if comp == use_components[0]:
                print(f"(n={r['n']}) " + "─" * max(0, 55 - len(label)))

            flag = _flag(r['lmin_corr'], r['nneg_corr'])
            wc   = r['wm_cov']
            wr   = r['wm_corr']

            supp_str = f"  supp={wr['eff_supp']:.1f}" if wr else ""
            dom_str  = f"  dom={wr['dom_idx']}" if wr else ""

            print(
                f"  {comp:<3}: cov λ_min={r['lmin_cov']:+.2e}  n_neg={r['nneg_cov']}"
                f"  │  corr λ_min={r['lmin_corr']:+.2e}  n_neg={str(r['nneg_corr']):>4}"
                f"{supp_str}{dom_str}{flag}"
            )

            # flag non-trivial dup_diag
            if r['dup_diag']:
                print(f"       dup diag(cov): {r['dup_diag'][:5]}"
                      f"{'…' if len(r['dup_diag'])>5 else ''}")
            if r['dup_dv']:
                print(f"       dup DV vals  : {r['dup_dv'][:5]}"
                      f"{'…' if len(r['dup_dv'])>5 else ''}")
            if r['corr_err']:
                print(f"       corr error   : {r['corr_err']}")

            summary_rows.append((label, r['n'], comp,
                                  r['lmin_cov'], r['nneg_cov'],
                                  r['lmin_corr'], r['nneg_corr']))

        all_results[label] = res

    # ── Final summary table ───────────────────────────────────────────────────
    comps = list(use_components)
    col_w = 14

    # build header
    hdr_comp = "  ".join(f"{'λ_min(corr)':>{col_w}}  {'n_neg':>5}" for _ in comps)
    print("\n\n" + "═" * 90)
    print("  FINAL SUMMARY  (correlation-matrix eigenvalues)")
    print("═" * 90)
    print(f"  {'Probe set':<28}  {'n':>5}  " +
          "  ".join(f"{'── '+c+' ──':>{col_w+8}}" for c in comps))
    print(f"  {'':28}  {'':5}  " +
          "  ".join(f"{'λ_min(corr)':>{col_w}}  {'n_neg':>5}" for _ in comps))
    print("  " + "─" * 86)

    # group rows by label
    by_label = {}
    for row in summary_rows:
        lbl = row[0]
        by_label.setdefault(lbl, {})[row[2]] = row

    for lbl in ["+".join(ps) for ps in probe_sets]:
        d = by_label.get(lbl, {})
        n_str = str(d[comps[0]][1]) if comps[0] in d else "?"
        cells = []
        for comp in comps:
            if comp in d:
                _, _, _, lc, nc, lr, nr = d[comp]
                neg = (nr > 0 or lr < 0) and not np.isnan(lr)
                nr_str  = f"{nr:>5}" if not np.isnan(nr) else "  nan"
                lr_str  = f"{lr:>+{col_w}.3e}" if not np.isnan(lr) else f"{'nan':>{col_w}}"
                mark    = " *" if neg else "  "
                cells.append(f"{lr_str}  {nr_str}{mark}")
            else:
                cells.append(f"{'—':>{col_w}}  {'—':>5}  ")
        print(f"  {lbl:<28}  {n_str:>5}  " + "  ".join(cells))

    print("  (* = non-PSD in correlation space)")
    print("═" * 90)

    return all_results


# ─────────────────────────────────────────────────────────────────────────────
# NG block-level diagnostic
# ─────────────────────────────────────────────────────────────────────────────

def diagnose_ng_blocks(
    probes_arr,
    cov_obj,
    diag_utils,
    eig_tol=0.0,
    warn_missing=False,
):
    """
    For a multi-probe set, decompose the NG covariance into probe-pair
    sub-blocks and report the PSD status of each.

    For diagonal blocks  (probe_A, probe_A): check that block alone.
    For off-diagonal pairs (probe_A, probe_B): assemble the 2×2 block matrix
        [[C_AA, C_AB], [C_BA, C_BB]]
    and check its PSD — this is the tightest test of cross-block consistency.

    Also reports the ℓ-level 2×2 Schur complement for 2-probe inputs so you
    can see at which ℓ the inconsistency first appears.
    """
    cov_ng, count_per_probe, nell = _build_ng(
        cov_obj, diag_utils, probes_arr, warn_missing=warn_missing)
    cov_ng = _sym(cov_ng)

    block_sizes = [int(c) * int(nell) for c in count_per_probe]
    edges       = np.concatenate([[0], np.cumsum(block_sizes)]).astype(int)
    n_probes    = len(probes_arr)
    label       = "+".join(probes_arr)

    print(f"\n{'═'*72}")
    print(f"  NG block diagnostic — probes: {label}  (nell={nell})")
    print(f"  Block sizes: { {p: s for p, s in zip(probes_arr, block_sizes)} }")
    print(f"{'═'*72}")
    print(f"\n  {'Block':<26}  {'n':>5}  {'λ_min(cov)':>13}  {'n_neg':>5}"
          f"  {'λ_min(corr)':>13}  {'n_neg':>5}  {'||B||_F':>11}")
    print("  " + "─" * 85)

    block_results = {}

    for i in range(n_probes):
        for j in range(i, n_probes):
            si, ei = edges[i], edges[i+1]
            sj, ej = edges[j], edges[j+1]

            if i == j:
                block = cov_ng[si:ei, si:ei]
                blabel = f"({probes_arr[i]}, {probes_arr[i]})"
            else:
                B_ii = cov_ng[si:ei, si:ei]
                B_jj = cov_ng[sj:ej, sj:ej]
                B_ij = cov_ng[si:ei, sj:ej]
                B_ji = cov_ng[sj:ej, si:ei]
                block  = np.block([[B_ii, B_ij], [B_ji, B_jj]])
                blabel = f"({probes_arr[i]},{probes_arr[j]}) joint"

            block = _sym(block)
            ev, _, lmin, nneg = _eig_stats(block, eig_tol)
            frob = float(np.linalg.norm(block, "fro"))

            try:
                corr  = _cov_to_corr(block)
                ev_r, _, lmin_r, nneg_r = _eig_stats(corr, eig_tol)
            except Exception:
                lmin_r, nneg_r = np.nan, -1

            flag = "  ← !" if (nneg_r > 0 or lmin_r < 0) else ""

            print(f"  {blabel:<26}  {block.shape[0]:>5}"
                  f"  {lmin:>13.3e}  {nneg:>5}"
                  f"  {lmin_r:>13.3e}  {nneg_r:>5}"
                  f"  {frob:>11.3e}{flag}")

            block_results[blabel] = dict(
                block=block, lmin_cov=lmin, nneg_cov=nneg,
                lmin_corr=lmin_r, nneg_corr=nneg_r, frob=frob,
            )

    # ── ℓ-level Schur complement (only for exactly 2 probes) ─────────────────
    if n_probes == 2:
        n0, n1 = block_sizes
        bins0   = n0 // nell
        bins1   = n1 // nell

        print(f"\n  ℓ-level Schur complement  [{probes_arr[0]}] in [{probes_arr[1]}] space")
        print(f"  (S_ℓ = C_AA(ℓ) − C_AB(ℓ) C_BB(ℓ)⁻¹ C_BA(ℓ), "
              f"min eigenvalue per ℓ bin)")
        print(f"\n  {'ℓ-bin':>6}  {'λ_min(Schur)':>14}  {'n_neg':>6}")
        print("  " + "─" * 32)

        schur_lmins = []
        for il in range(nell):
            # extract the (bins0 × bins0) and (bins1 × bins1) ℓ-level blocks
            idx0 = np.array([i * nell + il for i in range(bins0)])
            idx1 = np.array([i * nell + il for i in range(bins1)])
            # shift idx1 into the second block
            idx1_full = idx1 + n0

            A  = cov_ng[np.ix_(idx0, idx0)]
            B  = cov_ng[np.ix_(idx0, idx1_full)]
            C_ = cov_ng[np.ix_(idx1_full, idx1_full)]

            try:
                Cinv = np.linalg.inv(C_)
                S    = A - B @ Cinv @ B.T
                ev_s = np.linalg.eigvalsh(_sym(S))
                lmin_s = float(ev_s.min())
                nneg_s = int((ev_s < -eig_tol).sum())
            except Exception:
                lmin_s, nneg_s = np.nan, -1

            schur_lmins.append(lmin_s)
            flag = "  ← !" if nneg_s > 0 or lmin_s < 0 else ""
            print(f"  {il:>6}  {lmin_s:>14.3e}  {nneg_s:>6}{flag}")

        block_results["schur_lmins"] = schur_lmins

    print()
    return block_results


In [6]:
probe_sets = [
    ['gy'],
    ['gg'],
    ['ky'],
    ['kk'],
    ['yy'],
    ['gk'],
    ['yy','gy'],
    ['gy', 'gg'],
    ['ky', 'gy'],
    ['ky', 'kk'],
    ['yy', 'gk'],
    ['ky', 'kk', 'gy', 'gg'],
    ['ky', 'kk', 'gy', 'gg', 'gk', 'yy'],
]

all_res = diagnose_many_probe_sets(
    probe_sets=probe_sets,
    cov_obj=cov_test,
    diag_utils=diag_utils,
    use_components=('tot', 'G', 'NG'),
    warn_missing=True,
    eig_tol=1e-44,
    dup_atol=0.0,
    dup_rtol=0.0,
    show_matrix=False,
    show_plots=False,
)


── gy (n=75) ─────────────────────────────────────────────────────
  tot: cov λ_min=+1.17e-30  n_neg=0  │  corr λ_min=+4.47e-02  n_neg=   0  supp=2.7  dom=14
  G  : cov λ_min=+7.11e-31  n_neg=0  │  corr λ_min=+4.38e-01  n_neg=   0  supp=2.7  dom=15
  NG : cov λ_min=+2.86e-39  n_neg=0  │  corr λ_min=+4.61e-12  n_neg=   0  supp=6.1  dom=47

── gg (n=75) ─────────────────────────────────────────────────────
  tot: cov λ_min=+3.87e-20  n_neg=0  │  corr λ_min=+1.70e-03  n_neg=   0  supp=2.0  dom=13
  G  : cov λ_min=+2.04e-20  n_neg=0  │  corr λ_min=+7.18e-01  n_neg=   0  supp=2.0  dom=17
  NG : cov λ_min=+1.11e-30  n_neg=0  │  corr λ_min=+5.53e-14  n_neg=   0  supp=3.5  dom=1

── ky (n=75) ─────────────────────────────────────────────────────
  tot: cov λ_min=+7.96e-33  n_neg=0  │  corr λ_min=+1.52e-01  n_neg=   0  supp=2.0  dom=45
  G  : cov λ_min=+7.96e-33  n_neg=0  │  corr λ_min=+2.06e-01  n_neg=   0  supp=2.0  dom=45
  NG : cov λ_min=+5.95e-44  n_neg=0  │  corr λ_min=+1.12e-13  n_neg= 

In [ ]:
# ─── NG block diagnostic ──────────────────────────────────────────────────
# Isolates which off-diagonal NG block drives the non-PSD violation.
# Start with ky+kk (no galaxy probes, no P_sup ambiguity) then gy+gg.

print('=' * 60)
print('NG block diagnostic: [ky, kk]')
print('=' * 60)
ng_res_kykk = diagnose_ng_blocks(
    probes_arr=['ky', 'kk'],
    cov_obj=cov_test,
    diag_utils=diag_utils,
    eig_tol=1e-44,
)

print()
print('=' * 60)
print('NG block diagnostic: [gy, gg]')
print('=' * 60)
ng_res_gygg = diagnose_ng_blocks(
    probes_arr=['gy', 'gg'],
    cov_obj=cov_test,
    diag_utils=diag_utils,
    eig_tol=1e-14,
)

print()
print('=' * 60)
print('NG block diagnostic: [ky, kk, gy, gg]')
print('=' * 60)
ng_res_full = diagnose_ng_blocks(
    probes_arr=['ky', 'kk', 'gy', 'gg'],
    cov_obj=cov_test,
    diag_utils=diag_utils,
    eig_tol=1e-14,
)


NG block diagnostic: [ky, kk]

════════════════════════════════════════════════════════════════════════
  NG block diagnostic — probes: ky+kk  (nell=15)
  Block sizes: {'ky': 75, 'kk': 225}
════════════════════════════════════════════════════════════════════════

  Block                           n     λ_min(cov)  n_neg    λ_min(corr)  n_neg      ||B||_F
  ─────────────────────────────────────────────────────────────────────────────────────
  (ky, ky)                       75     -1.993e-45      0      6.888e-15      0    8.249e-27
  (ky,kk) joint                 300     -3.096e-37     38     -5.962e-15     26    7.356e-21  ← !
  (kk, kk)                      225     -2.306e-37     19     -5.687e-15     17    7.356e-21  ← !

  ℓ-level Schur complement  [ky] in [kk] space
  (S_ℓ = C_AA(ℓ) − C_AB(ℓ) C_BB(ℓ)⁻¹ C_BA(ℓ), min eigenvalue per ℓ bin)

   ℓ-bin    λ_min(Schur)   n_neg
  ────────────────────────────────
       0       4.277e-32       0
       1       3.289e-32       0
       2   

In [ ]:
# Check symmetry of NG covariance for all probe sets
import numpy as np

probe_sets_check = [
    ['gy'], ['gg'], ['ky'], ['kk'],
    ['gy', 'gg'], ['ky', 'kk'], ['ky', 'gy'],
    ['ky', 'kk', 'gy', 'gg'],
]

print(f"{'Probes':<30}  {'max|C-Cᵀ|':>12}  {'max|C|':>12}  {'rel asym':>10}")
print('-' * 70)
for ps in probe_sets_check:
    cov_ng, _, _, _ = diag_utils.build_cov_matrix(
        cov_test.covNG_dict, cov_test.Cl_result_dict, ps, warn_missing=False
    )
    C = np.asarray(cov_ng, dtype=float)
    asym = np.max(np.abs(C - C.T))
    scale = np.max(np.abs(C))
    rel = asym / scale if scale > 0 else 0.0
    label = '+'.join(ps)
    print(f"{label:<30}  {asym:>12.3e}  {scale:>12.3e}  {rel:>10.2e}")
